In [ ]:
import random
def a_star(coordinates):
    #just chatpgt this part because already made more complex A* in the past - no point remaking
    return random.randint(1, 7)

interesting_nodes = [[0, 1], [2, 4], [6, 7], [2, 10]]
print(interesting_nodes)

def distance_matrix_maker(interesting_nodes):

    num_inp = len(interesting_nodes)
    distance_matrix = []
    for i in range(num_inp):    #creates emoty distance matrix
        distances = []
        for j in range(num_inp):
            distances.append(0)
        distance_matrix.append(distances)
    print(distance_matrix)

    paths = []
    paths_index = []
    distances = []

    for i in range(num_inp-1):
        for j in range(num_inp-1-i):
            paths.append([interesting_nodes[i], interesting_nodes[num_inp-1-j]]) #creates nested list of each of the paths from one node to another
            paths_index.append([i, num_inp-1-j])

    print(paths)

    for nodes in paths:
        distances.append(a_star(nodes))

    print(distances)
    print(" ")
    print("l", paths_index)

    for indexes in range(len(paths_index)):   #updates distance matrix
        distance_matrix[paths_index[indexes][0]][paths_index[indexes][1]] = distances[indexes]
        distance_matrix[paths_index[indexes][1]][paths_index[indexes][0]] = distances[indexes]






    for row in distance_matrix:
        print(row)


    return distance_matrix

distance_matrix_maker(interesting_nodes)

In [ ]:
#single ant ACO iteration
nodes = ["a", "b", "c", "d"]
p = 0.5
start = "a"
from itertools import permutations
import random
import json
import requests
import math

def heuristic(lat1, lon1, lat2, lon2):
    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lon2 = math.radians(lon2)  # converts lattitudes and longitudes to radians
    lat2 = math.radians(lat2)
    equation = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2  # harvesine equation
    equation = math.sqrt(equation)
    equation = math.asin(equation)
    distance = 2 * 6371 * equation
    hofn = distance * 7  # this is an optimisation step. we are stating the importance of the heuristic cost, so that it is weighted specificallt agiasnt
    return hofn  # returns distance after equation

def geolocation(ip):
    key = "YOUR_TFL_API_KEY" #removing my personal key for security reasons
    url = "https://api.ipgeolocation.io/ipgeo?apiKey=" + key + "&ip=" + ip  # start and goal must be naptanIDS, this is the
    response = requests.get(url)  # request the data
    data = json.loads(response.text)  # put it in a more easily read format
    geolocation = {}
    geolocation["lat"] = float(data["latitude"])
    geolocation["lon"] = float(data["longitude"])
    return geolocation

def distance_matrix_maker():

  num_inp = int(input("How many IP addresses would you like to input: "))
  distance_matrix = []
  for i in range(num_inp):
    distances = []
    for j in range(num_inp):
      distances.append(0)
    distance_matrix.append(distances)

  print(distance_matrix)

  ips = []

  for i in range(num_inp):
    ip = input("Enter IP: ")
    ips.append(ip)

  for i in range(len(ips) - 1):
    geolocation1 = geolocation(ips[i])
    geolocation2 = geolocation(ips[i + 1])
    distance = heuristic(geolocation1["lat"], geolocation1["lon"], geolocation2["lat"], geolocation2["lon"])
    print(distance)
    distance_matrix[i][i + 1] = distance
    distance_matrix[i + 1][i] = distance

  geolocation1 = geolocation(ips[0])
  geolocation2 = geolocation(ips[-1])
  distance = heuristic(geolocation1["lat"], geolocation1["lon"], geolocation2["lat"], geolocation2["lon"])
  print(distance)
  distance_matrix[0][-1] = distance
  distance_matrix[-1][0] = distance

  return distance_matrix

costs = distance_matrix_maker()   #distance matrix declared globally as costs
print(costs)

def all_routes_finder(nodes):
  start_node = nodes[0]
  routes = []

  for perm in permutations(nodes[1:]):
    route = [start_node]+list(perm)+[start_node]
    routes.append(route)
  return routes   #returns 2D list of all the possible routes


routes = all_routes_finder(nodes)


def distance(start, end):    #works out distance between  2 nodes based off of a cost matrix
  global costs

  for node in nodes:
    if node == start:    #use index of list values to locate distance in distance matrix
      x = nodes.index(node) # if the start node chosen == the node we are on in the list search
    if node == end:
      y = nodes.index(node)
  distance = costs[x][y]
  return distance


def cost_matrix_maker(routes_find):#finds cost for each of the routes  #would use google API to find the shortest route between 2 places
  routes = {}
  counter = 0
  for route in routes_find:
    cost = 0
    for i in range(len(route)-1):
      cost += distance(route[i], route[i+1]) #finding sum of the costs of each of the distances from each of the nodes on each of the routes

    counter +=1
    routes[counter] = {"route": route, "distance": cost}
  return routes


routes_dict = cost_matrix_maker(routes)

pheramone_matrix = [[0, 0.10591133004926108, 0.06674082313681869, 0.10368663594470046], [0.10591133004926108, 0, 0.10368663594470046, 0.06674082313681869], [0.06674082313681869, 0.10368663594470046, 0, 0.10591133004926108], [0.10368663594470046, 0.06674082313681869, 0.10591133004926108, 0]]


def pheramone_matrix2(routes):   #

  global pheramone_matrix  # Declare pheramone_matrix as global
  global costs

  if 'pheramone_matrix' not in globals():  #as pheramone levels may alreayd exist, we want it to be cumulative
      pheramone_matrix = costs
      for row in range(len(pheramone_matrix)):  # creates empty matrix with all costs as 0
        for col in range(len(pheramone_matrix[row])):
          pheramone_matrix[row][col] = 0

  print(pheramone_matrix)
  for val in routes:
    pheramone_cost = 1 / (routes[val]["distance"])

    for i in range(len(routes[val]["route"]) - 1):
      pheramone_matrix[nodes.index(routes[val]["route"][i])][nodes.index(routes[val]["route"][i + 1])] += pheramone_cost * (1 - p)  # sum of the pheramone costs
      pheramone_matrix[nodes.index(routes[val]["route"][i + 1])][nodes.index(routes[val]["route"][i])] += pheramone_cost * (1 - p)
  return pheramone_matrix

pheramone_matrix = pheramone_matrix2(routes_dict)

#at this point pheramone matrix, distance matrix is made - no input needed

print("pheramone matrix", pheramone_matrix)
print("distance matrix", costs)
print("routes", routes)


def cumulative_freq_table(current_node):
  def probability_numerator(start, end):  # finds numerator value
    for node in nodes:
      if node == start:  # use index of list values to locate distance in distance matrix
        x = nodes.index(node)  # if the start node chosen == the node we are on in the list search
      if node == end:
        y = nodes.index(node)
    pheramone_cost = pheramone_matrix[x][y]
    distance = costs[x][y]
    numerator = pheramone_cost * 1 / distance

    return numerator

  def probability(start, end):
    numerator = probability_numerator(start, end)

    denominator = 0
    for node in nodes:  # goes through list of nodea and find probabilities of each path using probability_numerator() function
      if node != start:
        denominator += probability_numerator(start, node)

    probability = numerator / denominator
    return probability

  cft = {}  # stands for cumulative_freq_table
  counter = 0
  for node in nodes:
    if node != current_node:
      temp = (current_node, "to", node)
      temp = str(temp)
      print("current_node", current_node)
      print("node", node)
      cft[temp] = probability(current_node, node) + counter
      counter += probability(current_node, node)

  print("cumulative_freq_table", cft)
  return cft

def route_taken(cft):

  num = random.randint(1, 1000) / 1000
  print(num)

  for val in cft:
    print(val, cft[val])
    if num < cft[val]:
      node = val[-3]
      print("node in route taken", node)
      return node




def ant():
  cft = cumulative_freq_table("a")
  current_node = start
  run_loop = True
  while current_node != start or run_loop == True:
    pheramone_matrix = pheramone_matrix2(routes_dict)
    run_loop = False
    cft = cumulative_freq_table(current_node)
    current_node = route_taken(cft)
    print("cn", current_node)
    print("pheramone_matrix", pheramone_matrix)

ant()


def highest_pheromone_path(start_node_index, pheromone_matrix):
    path = [nodes[start_node_index]]  # Start at the initial node
    visited = set([start_node_index])  # Track visited nodes
    current_node = start_node_index  # Start from the initial node

    while len(visited) < len(nodes):
      # Get pheromone levels for current node, ignoring self-connections
      pheromone_levels = pheromone_matrix[current_node]

      # Find the unvisited neighbor with the highest pheromone level
      max_pheromone = -1
      next_node = None
      for i in range(len(pheromone_levels)):
        if i not in visited and pheromone_levels[i] > max_pheromone:
          max_pheromone = pheromone_levels[i]
          next_node = i

      # Move to the next node and mark it as visited
      if next_node is not None:
        path.append(nodes[next_node])
        visited.add(next_node)
        current_node = next_node
      else:
        break

    # Return to the start node to complete the path
    path.append(nodes[start_node_index])
    return path

# Start from node 'a' (index 0)
start_node_index = 0
result_path = highest_pheromone_path(start_node_index, pheramone_matrix)
print("Path with highest pheromone at each step:", result_path)

In [ ]:
def distance_matrix_maker():

  num_inp = 4
  distance_matrix = []
  for i in range(num_inp):
    distances = []
    for j in range(num_inp):
      distances.append(0)
    distance_matrix.append(distances)

  print(distance_matrix)

  ips = ["1.1.1.1", "2.2.2.2", "3.3.3.3", "4.4.4.4"]



  for i in range(len(ips) - 1):
    geolocation1 = geolocation(ips[i])
    geolocation2 = geolocation(ips[i + 1])
    distance = heuristic(geolocation1["lat"], geolocation1["lon"], geolocation2["lat"], geolocation2["lon"])
    print(distance)
    print(i)
    distance_matrix[i][i + 1] = distance   #issue here somewhere
    distance_matrix[i + 1][i] = distance
    print(distance_matrix)

  print(" ")
  geolocation1 = geolocation(ips[0])
  geolocation2 = geolocation(ips[-1])
  distance = heuristic(geolocation1["lat"], geolocation1["lon"], geolocation2["lat"], geolocation2["lon"])
  print(distance)
  distance_matrix[0][-1] = distance
  distance_matrix[-1][0] = distance

  return distance_matrix

costs = distance_matrix_maker()   #distance matrix declared globally as costs
for c in costs:
  print(c)

[[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
79777.39653188447
0
[[0, 79777.39653188447, 0, 0], [79777.39653188447, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
27103.246090833192
1
[[0, 79777.39653188447, 0, 0], [79777.39653188447, 0, 27103.246090833192, 0], [0, 27103.246090833192, 0, 0], [0, 0, 0, 0]]
10471.788436758297
2
[[0, 79777.39653188447, 0, 0], [79777.39653188447, 0, 27103.246090833192, 0], [0, 27103.246090833192, 0, 10471.788436758297], [0, 0, 10471.788436758297, 0]]
 
96745.21556963291
[0, 79777.39653188447, 0, 96745.21556963291]
[79777.39653188447, 0, 27103.246090833192, 0]
[0, 27103.246090833192, 0, 10471.788436758297]
[96745.21556963291, 0, 10471.788436758297, 0]
